In [1]:
import numpy as np
import pandas as pd


class ItemBasedCF:

    def __init__(
        self,
        item_user_matrix,
        top_k=20,
        min_common=10,
        significance_threshold=50,
        minimum_rated_neighbors=5,
        minimum_similarity=0.2
    ):

        self.item_user = item_user_matrix

        self.top_k = top_k
        self.min_common = min_common
        self.significance_threshold = significance_threshold
        self.minimum_rated_neighbors = minimum_rated_neighbors
        self.minimum_similarity = minimum_similarity
        
        # Cache
        self.movie_cache = {}
        
        
    # ==========================================================
    # Pearson Similarity
    # ==========================================================

    def pearson_similarity(
        self,
        vector1,
        vector2
    ):

        common = vector1.notna() & vector2.notna()

        common_count = common.sum()

        if common_count < self.min_common:
            return 0.0, common_count

        v1 = vector1[common]
        v2 = vector2[common]

        v1 = v1 - v1.mean()
        v2 = v2 - v2.mean()

        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)

        if norm1 == 0 or norm2 == 0:
            return 0.0, common_count

        similarity = np.dot(v1, v2) / (norm1 * norm2)

        weight = (
            min(common_count, self.significance_threshold)
            / self.significance_threshold
        )

        similarity *= weight

        return similarity, common_count
    
    
    
    # ==========================================================
    # Similar Movies
    # ==========================================================

    def get_similar_movies(
        self,
        target_movie,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        # Cache
        if target_movie in self.movie_cache:
            return self.movie_cache[target_movie]

        target = self.item_user.loc[target_movie]

        neighbors = []

        for other_movie in self.item_user.index:

            if other_movie == target_movie:
                continue

            similarity, common = self.pearson_similarity(
                target,
                self.item_user.loc[other_movie]
            )

            if similarity < self.minimum_similarity:
                continue

            neighbors.append(
                (
                    other_movie,
                    similarity,
                    common
                )
            )

        neighbors.sort(
            key=lambda x: x[1],
            reverse=True
        )

        neighbors = neighbors[:top_k]

        self.movie_cache[target_movie] = neighbors

        return neighbors
    
    # ==========================================================
    # Predict Rating
    # ==========================================================

    def predict_rating(
        self,
        target_user,
        target_movie,
        top_k=None
    ):
        if target_movie not in self.item_user.index:
            return None
        
        if target_user not in self.item_user.columns:
            return None

        if top_k is None:
            top_k = self.top_k

        rated_neighbors = 0
        
        # Get similar movies
        similar_movies = self.get_similar_movies(
            target_movie,
            top_k
        )

        numerator = 0.0
        denominator = 0.0

        for movie_id, similarity, common in similar_movies:

            rating = self.item_user.loc[
                movie_id,
                target_user
            ]

            if pd.isna(rating):
                continue
            rated_neighbors += 1

            numerator += similarity * rating
            denominator += abs(similarity)

        if rated_neighbors < self.minimum_rated_neighbors:
            return None
        
        if denominator == 0:
            return None

        prediction = numerator / denominator

        prediction = np.clip(
            prediction,
            1,
            5
        )

        return float(prediction)
    
    
    # ==========================================================
    # Candidate Movies
    # ==========================================================

    def get_candidate_movies(self,target_user,top_k=None):

        watched_movies = (
            self.item_user[target_user]
            .dropna()
            .index
        )

        candidate_movies = set()

        for movie in watched_movies:

            neighbors = self.get_similar_movies(
                movie,
                top_k
            )

            for movie_id, similarity, common in neighbors:

                candidate_movies.add(movie_id)

        candidate_movies -= set(watched_movies)

        return list(candidate_movies)




    # ==========================================================
    # Recommend Movies
    # ==========================================================

    def recommend(
        self,
        target_user,
        top_n=10,
        top_k=None
    ):

        if top_k is None:
            top_k = self.top_k

        candidate_movies = self.get_candidate_movies(
            target_user,
            top_k
        )

        recommendations = []

        for movie in candidate_movies:

            prediction = self.predict_rating(
                target_user,
                movie,
                top_k
            )

            if prediction is None:
                continue

            recommendations.append(
                (
                    movie,
                    prediction
                )
            )

        recommendations.sort(
            key=lambda x: x[1],
            reverse=True
        )

        return pd.DataFrame(
            recommendations[:top_n],
            columns=[
                "movie_id",
                "predicted_rating"
            ]
        )
        
        
    # ==========================================================
    # Evaluate
    # ==========================================================
    
    def evaluate(
        self,
        test_df
    ):
    
        predictions = []
    
        squared_errors = []
    
        absolute_errors = []
    
        for row in test_df.itertuples(index=False):
        
            user = row.user_id
            movie = row.movie_id
            actual = row.rating
    
            if movie not in self.item_user.index:
                continue
            
            if user not in self.item_user.columns:
                continue
            
            predicted = self.predict_rating(
                user,
                movie
            )
    
            if predicted is None:
                continue
            
            error = abs(
                actual - predicted
            )
    
            predictions.append(
                (
                    user,
                    movie,
                    actual,
                    predicted,
                    error
                )
            )
    
            absolute_errors.append(error)
    
            squared_errors.append(error ** 2)
    
        prediction_df = pd.DataFrame(
            predictions,
            columns=[
                "user_id",
                "movie_id",
                "actual",
                "predicted",
                "error"
            ]
        )
    
        mae = np.mean(absolute_errors)
        rmse = np.sqrt(np.mean(squared_errors))
    
        return prediction_df, rmse, mae

In [2]:
train = pd.read_csv(
    "data/u1.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    "data/u1.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [3]:
train_matrix = train.pivot(
    index="movie_id",
    columns="user_id",
    values="rating"
)
train_matrix.shape

(1650, 943)

In [4]:
cf = ItemBasedCF(train_matrix)


In [5]:
cf.get_similar_movies(1)

[(928, np.float64(0.6004968101418855), np.int64(60)),
 (378, np.float64(0.5465466451220039), np.int64(52)),
 (482, np.float64(0.5016376355477867), np.int64(48)),
 (520, np.float64(0.48645447114518026), np.int64(51)),
 (132, np.float64(0.4653059261478349), np.int64(128)),
 (72, np.float64(0.45979212878847026), np.int64(74)),
 (67, np.float64(0.45729177922384767), np.int64(52)),
 (498, np.float64(0.45704567008912733), np.int64(69)),
 (596, np.float64(0.4379854968523535), np.int64(72)),
 (465, np.float64(0.436678752216402), np.int64(46)),
 (969, np.float64(0.4342931962229257), np.int64(37)),
 (627, np.float64(0.4286632968154755), np.int64(47)),
 (523, np.float64(0.42547114893350274), np.int64(74)),
 (926, np.float64(0.41932514781518715), np.int64(59)),
 (232, np.float64(0.4181017398539325), np.int64(53)),
 (164, np.float64(0.416092972841935), np.int64(70)),
 (588, np.float64(0.4122757270173288), np.int64(108)),
 (319, np.float64(0.4097623166704634), np.int64(52)),
 (230, np.float64(0.4096

In [6]:
neighbors = cf.get_similar_movies(1)

for movie, sim, common in neighbors[:10]:
    print(movie, round(sim,3), common)

928 0.6 60
378 0.547 52
482 0.502 48
520 0.486 51
132 0.465 128
72 0.46 74
67 0.457 52
498 0.457 69
596 0.438 72
465 0.437 46


In [7]:
cf = ItemBasedCF(train_matrix)

cf.predict_rating(
    target_user=1,
    target_movie=50
)

4.444125843019111

In [8]:
cf = ItemBasedCF(train_matrix)

candidate_movies = cf.get_candidate_movies(1)

len(candidate_movies)

341

In [9]:
cf = ItemBasedCF(train_matrix)

cf.recommend(1)

,movie_id,predicted_rating
0,129,5.000000
1,319,4.883459
2,100,4.800762
3,531,4.799023
4,658,4.788385
5,421,4.717391
6,1039,4.712645
7,170,4.676863
8,184,4.664515
9,90,4.649528


In [10]:
599 in train_matrix.index

False

In [11]:
599 in test.movie_id.unique()

True

In [12]:
cf = ItemBasedCF(train_matrix)

prediction_df, rmse, mae = cf.evaluate(test)

print("MAE :", mae)
print("RMSE:", rmse)

MAE : 0.7694915074976751
RMSE: 0.9827322655313379
